# Joining three tables and a table to itself

Practice notebook for the datafromzero SQL lesson **Joining three tables and a table to itself** (S22). The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Predict the rows of the chain

In [ ]:
con.sql("""
SELECT COUNT(*) AS joined_rows
FROM order_items AS oi
INNER JOIN orders AS o ON oi.order_id = o.order_id
INNER JOIN products AS p ON oi.product_id = p.product_id
INNER JOIN stores AS s ON o.store_id = s.store_id
""").show()

### Step 2. Rebuild E7

In [ ]:
con.sql("""
SELECT COUNT(*) AS lines,
       SUM(oi.quantity * p.unit_price_usd - oi.discount_usd) AS sales_usd
FROM order_items AS oi
INNER JOIN orders AS o ON oi.order_id = o.order_id
INNER JOIN products AS p ON oi.product_id = p.product_id
INNER JOIN stores AS s ON o.store_id = s.store_id
WHERE s.store_name = 'Campus'
  AND p.category = 'coffee'
  AND o.status = 'completed'
  AND o.ordered_at >= DATE '2026-03-01'
  AND o.ordered_at < DATE '2026-04-01'
""").show()

### Step 3. Check one customer by hand

In [ ]:
con.sql("""
SELECT o.order_id, p.product_name, oi.quantity,
       oi.quantity * p.unit_price_usd - oi.discount_usd AS line_amount_usd
FROM customers AS c
INNER JOIN orders AS o ON o.customer_id = c.customer_id
INNER JOIN order_items AS oi ON oi.order_id = o.order_id
INNER JOIN products AS p ON oi.product_id = p.product_id
WHERE c.customer_id = 144
ORDER BY oi.order_item_id
""").show()

In [ ]:
con.sql("""
SELECT SUM(oi.quantity * p.unit_price_usd - oi.discount_usd) AS spend_usd
FROM customers AS c
INNER JOIN orders AS o ON o.customer_id = c.customer_id
INNER JOIN order_items AS oi ON oi.order_id = o.order_id
INNER JOIN products AS p ON oi.product_id = p.product_id
WHERE c.customer_id = 144
""").show()

### Step 4. Join the table to itself

In [ ]:
con.sql("""
SELECT COUNT(*) AS pairs
FROM customers AS c
INNER JOIN customers AS r ON c.referred_by = r.customer_id
""").show()

### Step 5. Look at the pairs

In [ ]:
con.sql("""
SELECT c.customer_id AS customer,
       c.loyalty_tier AS customer_tier,
       r.customer_id AS referrer,
       r.loyalty_tier AS referrer_tier
FROM customers AS c
INNER JOIN customers AS r ON c.referred_by = r.customer_id
WHERE c.customer_id IN (6, 30, 34, 144)
ORDER BY c.customer_id
""").show()

### Step 6. Verify

In [ ]:
con.sql("""
SELECT COUNT(referred_by) AS referred_customers FROM customers
""").show()

## Exercise 1. Completed sales of each store

Given: the grain is a line, the sum is the line amount, and the first join is in place. Add the joins to `products` and `stores`, keep only the completed orders, group by the store name and sort by it. The three amounts should add up to E21's Sales, 22,749.25.

In [ ]:
# Start from this query and finish it in a con.sql(...) cell of your own:
#
# SELECT s.store_name,
#        SUM(oi.quantity * p.unit_price_usd - oi.discount_usd) AS sales_usd
# FROM order_items AS oi
# INNER JOIN orders AS o ON oi.order_id = o.order_id
# -- add the other two joins, the filter and the grouping here

In [ ]:
# Your finished query:

## Exercise 2. Referred customers in the same city

How many referred customers live in the same city as the customer who referred them? Before you run anything, write one sentence in the cell below: what one row of the join is, how many rows it has before any city condition, and which customers can never be in it. Then write the query.

In [ ]:
# My expected result, in words:
#

In [ ]:
# Your query:

## Exercise 3. Spot the flaw

A colleague wants each referred customer next to the tier of the customer who referred them and runs this query. Its row for customer 3 reads `3 | 6 | bronze`, but customer 6 was referred by customer 3, who is gold. Find out what went wrong and write the right query.

In [ ]:
con.sql("""
SELECT c.customer_id AS customer,
       r.customer_id AS referrer,
       r.loyalty_tier AS referrer_tier
FROM customers AS c
INNER JOIN customers AS r ON c.customer_id = r.referred_by
""").show()

In [ ]:
# Your corrected query: